# Lab 03 · Pandas for Data Manipulation and Cleaning
**CS-3105 Machine Learning Lab** · Lab Instructor: Engr. Muhammad Awais

**CLO 6** (C-4 Analyze, PLO 2 Problem Analysis): *Analyze real-world datasets by applying data manipulation and visualization techniques.*

**Data:** `pakistan_used_cars.csv`, 24,973 real used-car ads from OLX Pakistan (Open Data Pakistan, *Pakistan Used Cars*, 2021). The ads are real and contain the problems real data has. Two kinds of formatting problem were added for practice (some prices written as text such as `Rs 1,430,000`, some city names in the wrong case or with extra spaces).

**Rules:** no Python loops over rows · label every output, e.g. `print("T4e:", ...)` · do not edit the CSV by hand · Runtime → Restart and run all before you submit · push this notebook to `ML-Lab-<RollNo>/Lab03/Lab03.ipynb` by **Thursday 5 November 2026, 11:59 PM (PKT)**.

Name: ______  Roll No: ______

## Warm-up (not graded): the class example from the lecture
Run this cell and repeat any lecture example on this small table before you start the tasks. `clean` is the tidy version, `messy` has the six kinds of problem.

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.width', 140)

clean = pd.DataFrame({
    "Roll": ["SE-01", "SE-02", "SE-03", "SE-04", "SE-05", "SE-06", "SE-07", "SE-08"],
    "Name": ["Ali", "Ayesha", "Bilal", "Fatima", "Hamza", "Sana", "Usman", "Zainab"],
    "City": ["Muzaffarabad", "Mirpur", "Bagh", "Muzaffarabad", "Rawalakot", "Mirpur", "Kotli", "Muzaffarabad"],
    "Maths": [78, 91, 45, 85, 62, 73, 38, 88],
    "Physics": [65, 88, 52, 79, 48, 81, 41, 92],
    "Attendance": [92, 97, 70, 88, 75, 90, 60, 95]})

messy = pd.DataFrame({
    "Roll": ["SE-01", "SE-02", "SE-03", "SE-04", "SE-05", "SE-06", "SE-07", "SE-04", "SE-08"],
    "Name": ["Ali", "Ayesha", "Bilal", "Fatima", "Hamza", "Sana", "Usman", "Fatima", "Zainab"],
    "City": [" Muzaffarabad", "Mirpur", "Bagh", "MUZAFFARABAD", "Rawalakot", "Mirpur", "Kotli", "MUZAFFARABAD", "Muzafarabad"],
    "Maths": [78, 91, np.nan, 85, 62, 73, 380, 85, 88],
    "Physics": [65, 88, 52, 79, 48, 105, 41, 79, 92],
    "Attendance": ["92%", "97", "70", "88", "75%", "90", "60", "88", "95"]})
print(clean)

    Roll    Name          City  Maths  Physics  Attendance
0  SE-01     Ali  Muzaffarabad     78       65          92
1  SE-02  Ayesha        Mirpur     91       88          97
2  SE-03   Bilal          Bagh     45       52          70
3  SE-04  Fatima  Muzaffarabad     85       79          88
4  SE-05   Hamza     Rawalakot     62       48          75
5  SE-06    Sana        Mirpur     73       81          90
6  SE-07   Usman         Kotli     38       41          60
7  SE-08  Zainab  Muzaffarabad     88       92          95


In [2]:
# Warm-up (not graded): lecture examples on the small class table
print("Warm-up, Maths > 80:")
print(clean[clean['Maths'] > 80])
print("Warm-up, messy Maths mean (wrong because of 380 and a duplicate):", messy["Maths"].mean())


Warm-up, Maths > 80:
    Roll    Name          City  Maths  Physics  Attendance
1  SE-02  Ayesha        Mirpur     91       88          97
3  SE-04  Fatima  Muzaffarabad     85       79          88
7  SE-08  Zainab  Muzaffarabad     88       92          95
Warm-up, messy Maths mean (wrong because of 380 and a duplicate): 117.75


### Setup
Put `pakistan_used_cars.csv` in the same folder as this notebook. In Colab: click the folder icon on the left → Upload → choose the CSV.

## Task 1 · Load and first look
- **(a)** Read `pakistan_used_cars.csv` into `df` and print its shape.
- **(b)** Show the first 5 rows.
- **(c)** Print the column names.
- **(d)** Show the data type of every column.
- **(e)** Which column should be a number but is not? Show two of its values that explain why.

In [3]:
# T1: load and first look
# (a) read the CSV and print its shape
df = pd.read_csv("pakistan_used_cars.csv")
print("T1a: shape =", df.shape)

# (b) first 5 rows
print("T1b: first 5 rows")
print(df.head())

# (c) column names
print("T1c: columns =", list(df.columns))

# (d) data type of every column
print("T1d: data types")
print(df.dtypes)

# (e) a column that should be numeric but is not: Price
print("T1e: dtype of Price =", df["Price"].dtype)
rs_value = df.loc[df["Price"].str.contains("Rs"), "Price"].iloc[0]   # a value with text in it
plain_value = df.loc[~df["Price"].str.contains("Rs"), "Price"].iloc[0]  # a normal-looking value
print("T1e: two Price values ->", repr(rs_value), "and", repr(plain_value))
print("T1e: how many Price values contain 'Rs':", df["Price"].str.contains("Rs").sum())


T1a: shape = (24973, 9)
T1b: first 5 rows
    Brand Condition    Fuel  KMs Driven        Model    Price Registered City Transaction Type    Year
0  Toyota      Used  Diesel         1.0        Prado  2100000         Karachi             Cash  1997.0
1  Suzuki      Used  Petrol    100000.0        Bolan   380000         Karachi             Cash  2006.0
2  Suzuki      Used     CNG     12345.0        Bolan   340000         Karachi             Cash  1998.0
3  Suzuki      Used  Petrol     94000.0         Alto   535000         Karachi             Cash  2010.0
4  Toyota      Used  Petrol    100000.0  Corolla XLI  1430000         Karachi             Cash  2013.0
T1c: columns = ['Brand', 'Condition', 'Fuel', 'KMs Driven', 'Model', 'Price', 'Registered City', 'Transaction Type', 'Year']
T1d: data types
Brand                   str
Condition               str
Fuel                    str
KMs Driven          float64
Model                   str
Price                   str
Registered City         str
Tra

## Task 2 · Retrieve data
- **(a)** The `Brand` column (first 5 values).
- **(b)** The columns `Brand`, `Model`, `Price` together (first 5 rows).
- **(c)** The first row, using `iloc`.
- **(d)** The `Model` in the row with index 100, using `loc`.
- **(e)** How many cars are registered in `"Islamabad"`?
- **(f)** How many **Honda** cars run on **Hybrid** fuel?
- **(g)** Cars from 2018 or newer, newest first (Brand, Model, Year, Price; first 10).
- **(h)** Try `df[df["Price"] > 1_000_000]`. What happens and why? Answer in a Markdown cell.

In [4]:
# T2: retrieve data (on the raw, still-dirty df)
print("T2a: Brand (first 5)")
print(df["Brand"].head())

print("T2b: Brand, Model, Price (first 5 rows)")
print(df[["Brand", "Model", "Price"]].head())

print("T2c: first row with iloc")
print(df.iloc[0])

print("T2d: Model in row with index 100 =", df.loc[100, "Model"])

print("T2e: cars registered in 'Islamabad' (exact match) =", (df["Registered City"] == "Islamabad").sum())
print("T2e: (for comparison) after strip + title-case =",
      (df["Registered City"].str.strip().str.title() == "Islamabad").sum())

print("T2f: Honda cars with Hybrid fuel =", ((df["Brand"] == "Honda") & (df["Fuel"] == "Hybrid")).sum())

print("T2g: cars from 2018 or newer, newest first (first 10)")
print(df[df["Year"] >= 2018].sort_values("Year", ascending=False)[["Brand", "Model", "Year", "Price"]].head(10))

print("T2h: dtype of Price =", df["Price"].dtype)
try:
    df[df["Price"] > 1_000_000]
except TypeError as e:
    print("T2h: TypeError ->", e)


T2a: Brand (first 5)
0    Toyota
1    Suzuki
2    Suzuki
3    Suzuki
4    Toyota
Name: Brand, dtype: str
T2b: Brand, Model, Price (first 5 rows)
    Brand        Model    Price
0  Toyota        Prado  2100000
1  Suzuki        Bolan   380000
2  Suzuki        Bolan   340000
3  Suzuki         Alto   535000
4  Toyota  Corolla XLI  1430000
T2c: first row with iloc
Brand                Toyota
Condition              Used
Fuel                 Diesel
KMs Driven              1.0
Model                 Prado
Price               2100000
Registered City     Karachi
Transaction Type       Cash
Year                 1997.0
Name: 0, dtype: object
T2d: Model in row with index 100 = Mehran VX
T2e: cars registered in 'Islamabad' (exact match) = 958
T2e: (for comparison) after strip + title-case = 986
T2f: Honda cars with Hybrid fuel = 202
T2g: cars from 2018 or newer, newest first (first 10)
        Brand       Model    Year    Price
13795   Honda         Fit  2020.0  1950000
10940  Toyota      Probox  202

**T2h answer:** `df[df["Price"] > 1_000_000]` fails with `TypeError: '>' not supported between instances of 'str' and 'int'`. The reason is that `Price` is stored as **text**, not as a number. 750 ads write the price as `"Rs 435,000"` (with `Rs` and commas), and a single text value is enough to make pandas read the whole column as text. Python cannot compare a text value such as `"2100000"` with the number 1,000,000, so the comparison stops with an error. The fix is in Task 3: remove `"Rs"` and the commas with `.str.replace()` (type 2, formatting) and then convert with `.astype("int64")` (type 3, data type).

## Task 3 · Clean the data: the six types of problem
Fix the problems **in this order** (text first, then duplicates):
- **(a) Missing values:** count them per column. Drop ads with no `Brand` or no `Model`; fill missing `Registered City`, `Fuel` and `Transaction Type` with `"Unknown"`; leave `Year` and `KMs Driven` as NaN. Print how many rows were removed.
- **(b) Formatting:** remove extra spaces and fix the case of city names (`.str.strip().str.title()`); remove `"Rs"` and commas from `Price`.
- **(c) Data types:** convert `Price` to `int64` and `Year` to `Int64`.
- **(d) Inconsistent categories:** `"Sukkar"` → `"Sukkur"` and `"Corrolla Altis"` → `"Corolla Altis"`. Print the number of different cities before and after.
- **(e) Duplicates:** count exact duplicate ads, drop them, print the new row count.
- **(f) Invalid values:** set `KMs Driven` to NaN when it is below 100, above 1,000,000, or the placeholder 12345 / 123456. Drop ads with the fake prices 12,345,678 or 87,654,321, and Suzuki/Daihatsu ads above 5,000,000.
- **(g) Self-check:** your final shape must be **(20484, 9)**.

In [5]:
# T3: clean the data, six types, in order
raw_rows = len(df)

# ---- (a) Missing values ----
print("T3a: missing values per column (before)")
print(df.isna().sum())
before = len(df)
df = df.dropna(subset=["Brand", "Model"])                       # ad is useless without brand/model
print("T3a: rows removed (no Brand or no Model) =", before - len(df))
df = df.fillna({"Registered City": "Unknown", "Fuel": "Unknown", "Transaction Type": "Unknown"})  # text labels -> "Unknown"
# Year and KMs Driven are left as NaN on purpose
print("T3a: missing values per column (after)")
print(df.isna().sum())

# ---- (b) Formatting ----
print("T3b: different city spellings before strip/title =", df["Registered City"].nunique())
df["Registered City"] = df["Registered City"].str.strip().str.title()
df["Price"] = (df["Price"].str.replace("Rs", "", regex=False)
                          .str.replace(",", "", regex=False)
                          .str.strip())
print("T3b: different city spellings after strip/title =", df["Registered City"].nunique())
print("T3b: Price still text, sample =", df["Price"].head(3).tolist())

# ---- (c) Data types ----
df["Price"] = df["Price"].astype("int64")
df["Year"] = df["Year"].astype("Int64")        # whole numbers that allow NaN
print("T3c: dtypes")
print(df.dtypes)

# ---- (d) Inconsistent categories ----
print("T3d: number of different cities BEFORE replace =", df["Registered City"].nunique())
df["Registered City"] = df["Registered City"].replace({"Sukkar": "Sukkur"})
df["Model"] = df["Model"].replace({"Corrolla Altis": "Corolla Altis"})
print("T3d: number of different cities AFTER replace  =", df["Registered City"].nunique())

# ---- (e) Duplicates ----
print("T3e: exact duplicate ads =", df.duplicated().sum())
df = df.drop_duplicates()
print("T3e: rows after dropping duplicates =", len(df))

# ---- (f) Invalid values ----
km = df["KMs Driven"]
bad_km = (km < 100) | (km > 1_000_000) | km.isin([12345, 123456])
print("T3f: KMs Driven values set to NaN =", bad_km.sum())
print("T3f: mean KMs Driven before =", round(km.mean()), "| median before =", km.median())
df.loc[bad_km, "KMs Driven"] = np.nan
print("T3f: mean KMs Driven after  =", round(df["KMs Driven"].mean()), "| median after  =", df["KMs Driven"].median())

fake_price = df["Price"].isin([12_345_678, 87_654_321])
cheap_brand_too_dear = df["Brand"].isin(["Suzuki", "Daihatsu"]) & (df["Price"] > 5_000_000)
print("T3f: fake-price ads =", fake_price.sum(), "| Suzuki/Daihatsu above 5,000,000 =", cheap_brand_too_dear.sum())
print("T3f: mean Price before removing these =", round(df["Price"].mean()))
df = df[~(fake_price | cheap_brand_too_dear)]
print("T3f: mean Price after removing these  =", round(df["Price"].mean()))

# ---- (g) Self-check ----
print("T3g: final shape =", df.shape)
assert df.shape == (20484, 9), "shape is not (20484, 9): re-check the cleaning order"
print("T3g: self-check passed, shape is (20484, 9)")


T3a: missing values per column (before)
Brand               2137
Condition           2136
Fuel                2445
KMs Driven          2286
Model               2448
Price                  0
Registered City     4636
Transaction Type    2445
Year                2284
dtype: int64
T3a: rows removed (no Brand or no Model) = 2448
T3a: missing values per column (after)
Brand               0
Condition           0
Fuel                0
KMs Driven          0
Model               0
Price               0
Registered City     0
Transaction Type    0
Year                0
dtype: int64
T3b: different city spellings before strip/title = 106
T3b: different city spellings after strip/title = 62
T3b: Price still text, sample = ['2100000', '380000', '340000']
T3c: dtypes
Brand                   str
Condition               str
Fuel                    str
KMs Driven          float64
Model                   str
Price                 int64
Registered City         str
Transaction Type        str
Year            

## Task 4 · Analyse
- **(a)** New column `Price_lakh` = price in lakh rupees (÷ 100,000).
- **(b)** New column `Age` = 2020 − `Year`.
- **(c)** The 5 most common brands.
- **(d)** The 5 cities with the most ads.
- **(e)** Median `Price_lakh` for each fuel type, highest first.
- **(f)** Median `Price_lakh` of a **Corolla GLI** for each year 2010–2018. How much value does it lose per year?
- **(g)** The 5 most expensive ads. Does one of them look wrong? Why?
- **(h)** Number of ads and median price for Cash vs Installment/Leasing. Interpret in 1–2 lines.

In [6]:
# T4: analyse
# (a) price in lakh rupees
df["Price_lakh"] = df["Price"] / 100_000
print("T4a: Price vs Price_lakh")
print(df[["Price", "Price_lakh"]].head())

# (b) age of the car (reference year 2020)
df["Age"] = 2020 - df["Year"]
print("T4b: Year vs Age")
print(df[["Year", "Age"]].head())

# (c) 5 most common brands
print("T4c: top 5 brands")
print(df["Brand"].value_counts().head(5))

# (d) 5 cities with the most ads
print("T4d: top 5 cities (note: 'Unknown' is the filled missing values)")
print(df["Registered City"].value_counts().head(5))

# (e) median Price_lakh per fuel type, highest first
print("T4e: median Price_lakh per fuel type")
print(df.groupby("Fuel")["Price_lakh"].median().sort_values(ascending=False))

# (f) Corolla GLI median price for each year 2010-2018
gli = df[(df["Model"] == "Corolla GLI") & df["Year"].between(2010, 2018)]
gli_by_year = gli.groupby("Year")["Price_lakh"].agg(["count", "median"])
print("T4f: Corolla GLI median Price_lakh per year")
print(gli_by_year)
yearly_change = gli_by_year["median"].diff()
print("T4f: change from the previous year (lakh)")
print(yearly_change)
slope, intercept = np.polyfit(gli_by_year.index.astype(float), gli_by_year["median"], 1)
print("T4f: average change per year of age, straight-line fit (lakh) =", round(slope, 2))
print("T4f: average change per year, 2010 -> 2018 (lakh) =", round((gli_by_year["median"].iloc[-1] - gli_by_year["median"].iloc[0]) / 8, 2))
print("T4f: compound % per year, 2010 -> 2018 =",
      round(((gli_by_year["median"].iloc[-1] / gli_by_year["median"].iloc[0]) ** (1 / 8) - 1) * 100, 1))

# (g) the 5 most expensive ads
print("T4g: 5 most expensive ads")
print(df.nlargest(5, "Price_lakh")[["Brand", "Model", "Year", "KMs Driven", "Price_lakh", "Registered City"]])
lc = df[(df["Model"] == "Land Cruiser") & (df["Year"] == 2005)]["Price_lakh"]
print("T4g: Land Cruiser from 2005 -> ads =", len(lc), "| median Price_lakh =", lc.median())
surf = df[df["Model"] == "Surf"]["Price_lakh"]
print("T4g: Surf (all years) -> ads =", len(surf), "| median Price_lakh =", surf.median(), "| 2nd highest =", surf.nlargest(2).iloc[-1])
lc15 = df[(df["Model"] == "Land Cruiser") & (df["Year"] == 2015)]["Price_lakh"]
print("T4g: Land Cruiser from 2015 -> ads =", len(lc15), "| median Price_lakh =", lc15.median())

# (h) Cash vs Installment/Leasing
print("T4h: ads and median Price_lakh by transaction type")
print(df.groupby("Transaction Type")["Price_lakh"].agg(["count", "median"]))
print("T4h: share of ads that are Installment/Leasing (%) =", round((df["Transaction Type"] == "Installment/Leasing").mean() * 100, 1))
print("T4h: median Age by transaction type")
print(df.groupby("Transaction Type")["Age"].median())
print("T4h: Suzuki/Daihatsu share (%) by transaction type")
print((df["Brand"].isin(["Suzuki", "Daihatsu"])).groupby(df["Transaction Type"]).mean().mul(100).round(1))


T4a: Price vs Price_lakh
     Price  Price_lakh
0  2100000       21.00
1   380000        3.80
2   340000        3.40
3   535000        5.35
4  1430000       14.30
T4b: Year vs Age
   Year  Age
0  1997   23
1  2006   14
2  1998   22
3  2010   10
4  2013    7
T4c: top 5 brands
Brand
Suzuki      8621
Toyota      4535
Honda       3025
Daihatsu    2231
Nissan       694
Name: count, dtype: int64
T4d: top 5 cities (note: 'Unknown' is the filled missing values)
Registered City
Karachi       15461
Unknown        2040
Lahore         1601
Islamabad       777
Rawalpindi      119
Name: count, dtype: int64
T4e: median Price_lakh per fuel type


Fuel
Hybrid    16.2750
Diesel    13.0000
Petrol     8.7525
CNG        3.8500
LPG        1.5750
Name: Price_lakh, dtype: float64
T4f: Corolla GLI median Price_lakh per year
      count  median
Year               
2010     64  12.250
2011     86  13.325
2012     86  14.125
2013     74  14.950
2014     65  16.750
2015     96  17.900
2016    110  19.000
2017     82  21.000
2018    102  23.200


T4f: change from the previous year (lakh)
Year
2010      NaN
2011    1.075
2012    0.800
2013    0.825
2014    1.800
2015    1.150
2016    1.100
2017    2.000
2018    2.200
Name: median, dtype: float64
T4f: average change per year of age, straight-line fit (lakh) = 1.33
T4f: average change per year, 2010 -> 2018 (lakh) = 1.37
T4f: compound % per year, 2010 -> 2018 = 8.3
T4g: 5 most expensive ads
        Brand         Model  Year  KMs Driven  Price_lakh Registered City
19688  Toyota  Land Cruiser  2017     32222.0       670.0         Karachi
22246  Toyota  Land Cruiser  2005     97000.0       520.0       Islamabad
19716  Toyota  Land Cruiser  2015     22222.0       420.0       Islamabad
22323  Toyota          Surf  2007      3222.0       410.0         Unknown
22688  Toyota         Prius  2015     30000.0       357.5         Unknown
T4g: Land Cruiser from 2005 -> ads = 3 | median Price_lakh = 58.0
T4g: Surf (all years) -> ads = 34 | median Price_lakh = 19.9 | 2nd highest = 53.5
T4g: Land

### Written answers for Task 4

**T4f:** A Corolla GLI loses about **1.4 lakh (roughly Rs 137,000) of median value for every extra year of age**. The median price falls from 23.2 lakh for a 2018 car to 12.25 lakh for a 2010 car, which is 10.95 lakh over 8 years, or 1.37 lakh per year (a straight-line fit gives 1.33). In percentage terms that is about 8% of its value per year. The loss is not constant in rupees: between 2011 and 2013 the gap between neighbouring years is only about 0.8 to 1.1 lakh, while between 2016, 2017 and 2018 it is 2.0 to 2.2 lakh. This fits depreciation by a percentage of the price (a more expensive newer car loses more rupees) rather than by a fixed amount.

**T4g:** The most expensive ad is a 2017 Toyota Land Cruiser at **670 lakh (Rs 67 million)**. That is extreme but believable for a recent, high-end Land Cruiser (the 2015 ones have a median of 230 lakh), so I would flag it as an *outlier* and check it, not delete it. Two other ads in the top 5 look **wrong**. (1) The #2 ad, a **2005** Land Cruiser at **520 lakh**, is a 15-year-old car priced above the 2015 model (420 lakh) and about 9 times the median of the 2005 Land Cruisers in the data (58 lakh), so it is most likely an extra zero typed by mistake. (2) The #4 ad, a **2007 Surf at 410 lakh**, is about 20 times the median Surf price (19.9 lakh) and far above the next most expensive Surf (53.5 lakh). These values are possible, but they are so far from similar cars that they are almost certainly data-entry errors and should be checked before computing averages.

**T4h:** There are **19,634 Cash ads with a median price of 6.9 lakh**, and only **850 Installment/Leasing ads (4.1%) with a median price of 3.0 lakh**. At first sight installment cars look less than half as expensive, but that is suspicious: the installment cars are much *newer* (median age 4 years against 13 years for Cash), and newer cars normally cost *more*. The most likely explanation is that many installment ads show only the **down payment or advance** as the "Price", not the full price of the car. So the two medians are not comparable, and we should not conclude that cars sold on installment are cheaper. We would need the full price (or the down payment recorded separately) to compare them properly.

## Task 5 · Save your work
Save the cleaned DataFrame as `pakistan_used_cars_clean.csv` (Lab 04 uses it) and print its shape.

In [7]:
# T5: save the cleaned data
df.to_csv("pakistan_used_cars_clean.csv", index=False)
print("T5: saved cleaned df, shape =", df.shape)
print("T5: re-read check, shape =", pd.read_csv("pakistan_used_cars_clean.csv").shape)


T5: saved cleaned df, shape = (20484, 11)
T5: re-read check, shape = (20484, 11)


## Task 6 · Reflection (Markdown cell)
Write 3–5 sentences: which of the six cleaning problems would have given the most misleading answer if you had ignored it, and two things you learned about the used-car market in Pakistan.

**Reflection.** Ignoring the **invalid values (type 6)** would have misled me most, because it gives a wrong answer with no error message: the average kilometres driven was 131,160 with the impossible values (1 km, 123456 and so on) and 93,238 after setting them to NaN, a difference of about 38,000 km, and a few fake or mistyped prices (Rs 67 million for a Land Cruiser, Rs 87,654,321 for a Ravi) pull the average price up in the same silent way. The other problems were at least visible: text prices crashed the `Price > 1_000_000` comparison, but formatting (type 2) still silently gave 958 cars in Islamabad instead of 986, and the 2,028 duplicates would have inflated the counts. About the Pakistani used-car market, I learned that it is dominated by Suzuki (8,621 ads, then Toyota and Honda) and by Karachi (15,461 of 20,484 ads, ignoring the 2,040 with no city). I also learned that fuel type strongly affects price (median 16.3 lakh for Hybrid and 13 lakh for Diesel against only 3.85 lakh for CNG), and that a Corolla GLI loses about 1.4 lakh (around 8%) of its value every year.